# Step 9 (rebuilt) — Reporter

Redone against D-072. `policy_brief()` is retired — it ranked SKUs by a per-SKU `saving_eur` that no longer has a well-defined meaning under the joint per-class search (D-071). Two functions replace it:

- **`line_policy_brief()`** — one row per (line, class): the direct "what to set" table
- **`class_breakdown_view()`** — real per-class economics, summed from the actual joint simulation that produced the winning policy — not an illustrative split

`avoidable_cost_view`, `capacity_warning`, `owner_view`, `portfolio_brief` are unchanged — none depend on Step 7's output shape.

Prerequisite: Steps 4, 5a, 7 and 8 have already been run. This notebook checks for each artefact and names the producing step if missing.

## Setup

In [ ]:
import subprocess, os, sys
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r
REPO = '/content/ibp-tradeoff'
os.chdir('/content'); sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## Check upstream artefacts

In [ ]:
import pandas as pd, numpy as np, yaml
from src.reporter import require_artefact, ReporterViolation
NEEDED = ['data_primary/clean/clean_master.parquet', 'data_primary/clean/sku_master.parquet']
for f in NEEDED:
    try:
        require_artefact(f); print('OK  ', f)
    except ReporterViolation as e:
        print('MISSING', f); print(e)

## Rebuild Step 4, upload Step 5a and Step 7/8 outputs

In [ ]:
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')
ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv (Step 5a):')
uploaded = files.upload()
demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'flagged: {len(flagged)}')

In [ ]:
print('Select line_results.csv, sku_level.csv (Step 7) and '
      'step08_portfolio_summary.csv, step08_lever_consistency.csv (Step 8):')
uploaded2 = files.upload()
line_results            = pd.read_csv('line_results.csv')
sku_level                = pd.read_csv('sku_level.csv')
portfolio_summary       = pd.read_csv('step08_portfolio_summary.csv')
lever_consistency_table = pd.read_csv('step08_lever_consistency.csv')
print('line_results:', line_results.shape, '| sku_level:', sku_level.shape)

## Build the engine

In [ ]:
from src.engine import TradeOffEngine, LeverSettings, build_line_master
assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)
print('assumption fingerprint:', engine.assumption_fingerprint)

## Avoidable cost view — the Step 6 gate scenario

Unchanged from the original Step 9: both `total_avoidable_cost_eur` and `total_reported_cost_eur` are always returned together (D-062).

In [ ]:
from src.reporter import (avoidable_cost_view, owner_view, capacity_warning,
                          line_policy_brief, class_breakdown_view,
                          portfolio_brief, export_artefacts)

MVD_LINE = 'L3'
cat = str(engine.sku_master.loc[engine.line_skus(MVD_LINE)[0], 'category'])
base = LeverSettings.defaults(assumptions, cat, 'A')
gate_scenario = engine.run_scenario(MVD_LINE, base)

avoidable = avoidable_cost_view(gate_scenario, assumptions, schema)
for k in ('lost_sales_eur','excess_obsolescence_eur','working_capital_cost_eur',
          'conversion_cost_avoidable_eur','conversion_cost_fixed_eur'):
    print(f'{k:<30}{avoidable[k]:>16,.0f}')
print(f'{"total_avoidable_cost_eur":<30}{avoidable["total_avoidable_cost_eur"]:>16,.0f}')
print(f'{"total_reported_cost_eur":<30}{avoidable["total_reported_cost_eur"]:>16,.0f}')

## Owner view

In [ ]:
owners = owner_view(avoidable, assumptions)
print(owners.to_string(index=False))

## Capacity warning — Step 8 reference (static)

In [ ]:
ps_indexed = portfolio_summary.set_index('line_id')
tests = [('L2', 10.0), ('L2', 4.0), ('L3', 10.0)]
for line_id, cover in tests:
    msg = capacity_warning(line_id, cover, ps_indexed.loc[line_id])
    print(f'{line_id} @ cover={cover}: {msg or "(no warning)"}')
print()
print('Note: Step 10s app additionally checks capacity LIVE for whatever')
print('scenario is on screen. This cell demonstrates the Step 8-based static')
print('reference check only.')

## Line policy brief — the "what to set" table (D-072)

Replaces the retired `policy_brief()`. One row per (line, class): the winning cover/service from Step 7's joint search, and that LINE's own saving_eur (not a per-SKU or per-class share of it — saving is a line-level fact).

In [ ]:
lpb = line_policy_brief(line_results)
print(lpb.round(3).to_string(index=False))

## Class breakdown — where the trade-off actually lands (D-072)

Real per-class economics, genuinely summed from `sku_level` (which came from the one real joint simulation that produced each line's winning policy). Conversion cost is shown once per line, not split — D-066's finding that it isn't separable per SKU still holds.

In [ ]:
cb = class_breakdown_view(sku_level)
print(cb.round(1).to_string(index=False))

## Portfolio brief

In [ ]:
brief_text = portfolio_brief(portfolio_summary, lever_consistency_table)
print(brief_text)

## Export the artefacts

In [ ]:
paths = export_artefacts(avoidable, lpb, cb, brief_text, out_dir='.')
print(paths)

## Tests

In [ ]:
sh('python -m pytest tests/test_reporter.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_policy_model.py '
  'tests/test_portfolio_sweep.py tests/test_pipeline.py -q --no-header')

## Consolidated report — the only cell to copy

In [ ]:
import hashlib, subprocess

t_rep  = subprocess.run('python -m pytest tests/test_reporter.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_eng  = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_pol  = subprocess.run('python -m pytest tests/test_policy_model.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_port = subprocess.run('python -m pytest tests/test_portfolio_sweep.py -q --no-header',
                        shell=True, capture_output=True, text=True)
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 9 (REBUILT) - REPORTER - CONSOLIDATED REPORT'); w('='*78)
w(f'assumption set   : {engine.assumption_fingerprint}')
w(f'engine.py sha    : {hashlib.sha256(open("src/engine.py","rb").read()).hexdigest()[:12]}')
w(f'reporter.py sha  : {hashlib.sha256(open("src/reporter.py","rb").read()).hexdigest()[:12]}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')

w(''); w('-- 1. AVOIDABLE COST VIEW - Step 6 gate scenario '+'-'*28)
for k in ('lost_sales_eur','excess_obsolescence_eur','working_capital_cost_eur',
          'conversion_cost_avoidable_eur','conversion_cost_fixed_eur',
          'total_avoidable_cost_eur','total_reported_cost_eur'):
    w(f'{k:<32}{avoidable[k]:>16,.0f}')

w(''); w('-- 2. OWNER VIEW '+'-'*61)
w(owners.to_string(index=False))

w(''); w('-- 3. LINE POLICY BRIEF (D-072) '+'-'*46)
w(lpb.round(3).to_string(index=False))

w(''); w('-- 4. CLASS BREAKDOWN (D-072) '+'-'*48)
w(cb.round(1).to_string(index=False))

w(''); w('-- 5. PORTFOLIO BRIEF '+'-'*56)
w(brief_text)

w(''); w('-- 6. TESTS '+'-'*66)
for label, r in (('test_reporter.py', t_rep), ('test_engine.py', t_eng),
                 ('test_policy_model.py', t_pol), ('test_portfolio_sweep.py', t_port),
                 ('test_pipeline.py', t_pipe)):
    w(f'{label:<24}: ' + (r.stdout.strip().splitlines() or ["no output"])[-1])
if any(r.returncode for r in (t_rep,t_eng,t_pol,t_port,t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_rep,t_eng,t_pol,t_port,t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 7. CHECKS '+'-'*65)
checks = [
 ('both avoidable and reported totals present',
  'total_avoidable_cost_eur' in avoidable and 'total_reported_cost_eur' in avoidable),
 ('reported total exceeds avoidable total', avoidable['total_reported_cost_eur'] > avoidable['total_avoidable_cost_eur']),
 ('line_policy_brief has 3 rows per line', len(lpb) == lpb.line_id.nunique() * 3),
 ('class_breakdown does not split conversion',
  bool(cb.groupby('line_id')['line_conversion_cost_eur'].nunique().eq(1).all())),
 ('owner mapping present and used', len(owners) > 0),
 ('all test suites pass', all(r.returncode==0 for r in (t_rep,t_eng,t_pol,t_port,t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step09_report.txt','w').write(report_text)
try:
    import shutil
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step09_report.txt','avoidable_cost_summary.csv',
                  'line_policy_brief.csv','class_breakdown.csv','portfolio_brief.txt'):
            if os.path.exists(f): shutil.copy(f, d)
        print('saved to', d, '\\n')
except Exception as e:
    print('Drive copy skipped:', e, '\\n')
print(report_text)